## Section 1: Setup + Test API

### create notebook and import library

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/inflation-nowcast-th'
os.makedirs(PROJECT_PATH, exist_ok=True)
os.makedirs(f'{PROJECT_PATH}/data', exist_ok=True)
os.makedirs(f'{PROJECT_PATH}/notebooks', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!pip install python-dotenv sqlalchemy -q

### BOT API

In [10]:
from google.colab import userdata
API_KEY = userdata.get('BOT_API_KEY')

In [11]:
import requests
from google.colab import userdata

API_KEY = userdata.get('BOT_API_KEY')

headers = {
    "Accept": "*/*",
    "Authorization": API_KEY
}

url = "https://gateway.api.bot.or.th/Stat-ExchangeRate/v2/DAILY_AVG_EXG_RATE/"

params = {
    "start_period": "2024-01-01",
    "end_period": "2024-01-31",
    "currency": "USD"
}

response = requests.get(url, headers=headers, params=params)
print("Status code:", response.status_code)
print(response.text[:1000])

Status code: 200
{"result":{"timestamp":"2026-09-17 00:44:26","api":"Daily Weighted-average Interbank Exchange Rate - THB / USD","data":{"data_header":{"report_name_eng":"Rates of Exchange of Commercial Banks in Bangkok Metropolis (2002-present)","report_name_th":"อัตราแลกเปลี่ยนเฉลี่ยของธนาคารพาณิชย์ในกรุงเทพมหานคร (2545-ปัจจุบัน)","report_uoq_name_eng":"(Unit: Baht / 1 Unit of Foreign Currency)","report_uoq_name_th":"(หน่วย: บาท ต่อ 1 หน่วยเงินตราต่างประเทศ)","report_source_of_data":[{"source_of_data_eng":"Bank of Thailand","source_of_data_th":"ธนาคารแห่งประเทศไทย"}],"report_remark":[{"report_remark_eng":"Since Nov 16, 2015 the data regarding Buying Transfer Rate of PKR has been changed to Buying Rate using Foreign Exchange Rates (LSEG) with Bangkok Market Crossing.","report_remark_th":"ตั้งแต่วันที่ 16 พ.ย. 2558 ข้อมูลในอัตราซื้อเงินโอนของสกุล PKR ได้เปลี่ยนเป็นอัตราซื้อที่ใช้อัตราในตลาดต่างประเทศ (LSEG) คำนวณผ่านอัตราซื้อขายเงินดอลลาร์ สรอ. ในตลาดกรุงเทพฯ"}],"last_updated":"2026-09

### test - extract data

In [12]:
import pandas as pd
df = pd.json_normalize(response.json())
df.to_csv(f'{PROJECT_PATH}/data/test_exchange_rate.csv', index=False)

In [13]:
import json

data = response.json()
print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

{
  "result": {
    "timestamp": "2026-09-17 00:44:26",
    "api": "Daily Weighted-average Interbank Exchange Rate - THB / USD",
    "data": {
      "data_header": {
        "report_name_eng": "Rates of Exchange of Commercial Banks in Bangkok Metropolis (2002-present)",
        "report_name_th": "อัตราแลกเปลี่ยนเฉลี่ยของธนาคารพาณิชย์ในกรุงเทพมหานคร (2545-ปัจจุบัน)",
        "report_uoq_name_eng": "(Unit: Baht / 1 Unit of Foreign Currency)",
        "report_uoq_name_th": "(หน่วย: บาท ต่อ 1 หน่วยเงินตราต่างประเทศ)",
        "report_source_of_data": [
          {
            "source_of_data_eng": "Bank of Thailand",
            "source_of_data_th": "ธนาคารแห่งประเทศไทย"
          }
        ],
        "report_remark": [
          {
            "report_remark_eng": "Since Nov 16, 2015 the data regarding Buying Transfer Rate of PKR has been changed to Buying Rate using Foreign Exchange Rates (LSEG) with Bangkok Market Crossing.",
            "report_remark_th": "ตั้งแต่วันที่ 16 พ.ย. 2558 ข้

In [14]:
def find_path(obj, target_key="period", path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}['{k}']"
            if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict) and target_key in v[0]:
                print("เจอ list ของข้อมูลรายวันที่:", new_path)
                return new_path
            result = find_path(v, target_key, new_path)
            if result:
                return result
    return None

data = response.json()
path_found = find_path(data)
print("Path:", path_found)

เจอ list ของข้อมูลรายวันที่: ['result']['data']['data_detail']
Path: ['result']['data']['data_detail']


In [15]:
records = data['result']['data']['data_detail']

df = pd.json_normalize(records)
print(df.shape)
df.head(10)

(22, 8)


,period,currency_id,currency_name_th,currency_name_eng,buying_sight,buying_transfer,selling,mid_rate
0,2024-01-31,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.1667000,35.2676000,35.5944000,35.4310000
1,2024-01-30,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.1020000,35.1932000,35.5197000,35.3565000
2,2024-01-29,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.3287000,35.4108000,35.7326000,35.5717000
3,2024-01-26,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.4425000,35.5180000,35.8382000,35.6781000
4,2024-01-25,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.5160000,35.5949000,35.9129000,35.7539000
5,2024-01-24,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.5382000,35.6340000,35.9589000,35.7965000
6,2024-01-23,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.3794000,35.4725000,35.7888000,35.6307000
7,2024-01-22,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.3126000,35.3894000,35.7079000,35.5487000
8,2024-01-19,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.3017000,35.3930000,35.7171000,35.5551000
9,2024-01-18,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),35.3220000,35.4070000,35.7223000,35.5647000


## Section 2: Exchange Rate Data

In [16]:
params = {
    "start_period": "2020-01-01",
    "end_period": "2024-12-31",
    "currency": "USD"
}

response = requests.get(url, headers=headers, params=params)

# Check the status code before processing the response
if response.status_code == 200:
    data = response.json()
    records = data['result']['data']['data_detail']
    df = pd.json_normalize(records)
    print(df.shape)
    df.head()
else:
    print(f"Error: {response.status_code} - {response.reason}")
    print("Response content:", response.text[:500])


Error: 400 - Bad Request
Response content: {"httpCode":"400","httpMessage":"Bad Request","moreInformation":[{"message":"Exceed limit period. Limit period is 31 days"}]}


In [17]:
import time

def fetch_bot_data(endpoint_url, start_period, end_period, extra_params=None):
    params = {"start_period": start_period, "end_period": end_period}
    if extra_params:
        params.update(extra_params)
    response = requests.get(endpoint_url, headers=headers, params=params)
    if response.status_code != 200:
        print(f"Error {start_period} - {end_period}: {response.status_code} - {response.text[:200]}")
        return None
    data = response.json()
    records = data['result']['data']['data_detail']
    return pd.json_normalize(records)

date_ranges = pd.date_range(start="2020-01-01", end="2026-09-01", freq="MS")

all_dfs = []
for i in range(len(date_ranges) - 1):
    start = date_ranges[i].strftime("%Y-%m-%d")
    end = (date_ranges[i+1] - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    df_month = fetch_bot_data(url, start, end, {"currency": "USD"})
    if df_month is not None:
        all_dfs.append(df_month)
        print(f"ดึงสำเร็จ: {start} ถึง {end} ({len(df_month)} แถว)")

    time.sleep(0.3)

df_full = pd.concat(all_dfs, ignore_index=True)
df_full = df_full.drop_duplicates(subset=["period", "currency_id"])
df_full = df_full.sort_values("period").reset_index(drop=True)

print("รวมทั้งหมด:", df_full.shape)
df_full.head()

ดึงสำเร็จ: 2020-01-01 ถึง 2020-01-31 (22 แถว)
ดึงสำเร็จ: 2020-02-01 ถึง 2020-02-29 (19 แถว)
ดึงสำเร็จ: 2020-03-01 ถึง 2020-03-31 (22 แถว)
ดึงสำเร็จ: 2020-04-01 ถึง 2020-04-30 (21 แถว)
ดึงสำเร็จ: 2020-05-01 ถึง 2020-05-31 (18 แถว)
ดึงสำเร็จ: 2020-06-01 ถึง 2020-06-30 (21 แถว)
ดึงสำเร็จ: 2020-07-01 ถึง 2020-07-31 (20 แถว)
ดึงสำเร็จ: 2020-08-01 ถึง 2020-08-31 (20 แถว)
ดึงสำเร็จ: 2020-09-01 ถึง 2020-09-30 (20 แถว)
ดึงสำเร็จ: 2020-10-01 ถึง 2020-10-31 (20 แถว)
ดึงสำเร็จ: 2020-11-01 ถึง 2020-11-30 (21 แถว)
ดึงสำเร็จ: 2020-12-01 ถึง 2020-12-31 (19 แถว)
ดึงสำเร็จ: 2021-01-01 ถึง 2021-01-31 (20 แถว)
ดึงสำเร็จ: 2021-02-01 ถึง 2021-02-28 (18 แถว)
ดึงสำเร็จ: 2021-03-01 ถึง 2021-03-31 (23 แถว)
ดึงสำเร็จ: 2021-04-01 ถึง 2021-04-30 (18 แถว)
ดึงสำเร็จ: 2021-05-01 ถึง 2021-05-31 (18 แถว)
ดึงสำเร็จ: 2021-06-01 ถึง 2021-06-30 (21 แถว)
ดึงสำเร็จ: 2021-07-01 ถึง 2021-07-31 (20 แถว)
ดึงสำเร็จ: 2021-08-01 ถึง 2021-08-31 (21 แถว)
ดึงสำเร็จ: 2021-09-01 ถึง 2021-09-30 (21 แถว)
ดึงสำเร็จ: 2021-10-01 ถึง 2021-10-

,period,currency_id,currency_name_th,currency_name_eng,buying_sight,buying_transfer,selling,mid_rate
0,2020-01-02,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),29.8406000,29.9315000,30.2826000,30.1071000
1,2020-01-03,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),29.8997000,29.9872000,30.3148000,30.1510000
2,2020-01-06,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),29.8793000,29.9709000,30.3316000,30.1513000
3,2020-01-07,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),29.8849000,29.9747000,30.3215000,30.1481000
4,2020-01-08,USD,สหรัฐอเมริกา : ดอลลาร์ (USD),USA : DOLLAR (USD),30.0220000,30.1115000,30.4548000,30.2832000


In [18]:
df_full.to_csv(f'{PROJECT_PATH}/data/exchange_rate_usd.csv', index=False)

## Section 3: Policy Rate Data

In [19]:
import pandas as pd

raw = pd.read_excel(
    f'{PROJECT_PATH}/data/table-mpc.xlsx',
    sheet_name='Sheet1',
    header=0,
    skiprows=[1]
)

raw.head()

,ครั้งที่,วันที่,Date,การประชุม,Meeting,มติ กนง.,Decision,อัตราดอกเบี้ยนโยบาย\nPolicy rate\n(%),จำนวนเสียงในการลงมติ3/ (Balance of votes3/),Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,ครั้งที่ 1/2543,2000-05-23 00:00:00,2000-05-23 00:00:00,กนง.,MPB,คง,Hold,1.5,NaN,NaN,NaN,NaN,NaN,NaN
1,ครั้งที่ 2/2543,2000-07-18 00:00:00,2000-07-18 00:00:00,กนง.,MPB,คง,Hold,1.5,NaN,NaN,NaN,NaN,NaN,NaN
2,ครั้งที่ 3/2543,2000-09-05 00:00:00,2000-09-05 00:00:00,กนง.,MPB,คง,Hold,1.5,NaN,NaN,NaN,NaN,NaN,NaN
3,ครั้งที่ 4/2543,2000-10-17 00:00:00,2000-10-17 00:00:00,กนง.,MPB,คง,Hold,1.5,NaN,NaN,NaN,NaN,NaN,NaN
4,ครั้งที่ 5/2543,2000-12-06 00:00:00,2000-12-06 00:00:00,กนง.,MPB,คง,Hold,1.5,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
print(raw.columns.tolist())

['ครั้งที่', 'วันที่', 'Date', 'การประชุม', 'Meeting', 'มติ กนง.', 'Decision', 'อัตราดอกเบี้ยนโยบาย\nPolicy rate\n(%)', 'จำนวนเสียงในการลงมติ3/ (Balance of votes3/)', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


In [21]:
df_policy = raw[['Date', 'อัตราดอกเบี้ยนโยบาย\nPolicy rate\n(%)']].copy()
df_policy.columns = ['announcement_date', 'policy_rate']

df_policy = df_policy.dropna(subset=['announcement_date', 'policy_rate']).reset_index(drop=True)
df_policy['announcement_date'] = pd.to_datetime(df_policy['announcement_date'])
df_policy['policy_rate'] = pd.to_numeric(df_policy['policy_rate'], errors='coerce')
df_policy = df_policy.dropna(subset=['policy_rate']).reset_index(drop=True)

print(df_policy.shape)
df_policy.tail(10)

(206, 2)


,announcement_date,policy_rate
196,2025-02-26,2.00
197,2025-04-30,1.75
198,2025-06-25,1.75
199,2025-08-13,1.50
200,2025-10-08,1.50
201,2025-12-17,1.25
202,2026-02-25,1.00
203,2026-04-29,1.00
204,2026-06-24,1.00
205,2026-08-26,1.00


In [22]:
df_policy.to_csv(f'{PROJECT_PATH}/data/policy_rate.csv', index=False)

## Section 4: CPI Data (สนค.)

In [23]:
import pandas as pd

raw = pd.read_excel(f'{PROJECT_PATH}/data/CPI-G_Report.xlsx', sheet_name='export', header=None)

header_row = raw.iloc[4]
data_rows = raw.iloc[5:].copy()
data_rows.columns = header_row
data_rows = data_rows.reset_index(drop=True)

cpi_headline = data_rows[data_rows['รหัส'].astype(str).str.strip() == '00000'].copy()
print("จำนวนปีที่เจอ:", cpi_headline.shape[0])
print(cpi_headline[['รหัส', 'หมวด', 'ปี']])

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


จำนวนปีที่เจอ: 7
4      รหัส          หมวด    ปี
0     00000  รวมทุกรายการ  2569
189   00000  รวมทุกรายการ  2568
378   00000  รวมทุกรายการ  2567
553   00000  รวมทุกรายการ  2566
728   00000  รวมทุกรายการ  2565
903   00000  รวมทุกรายการ  2564
1074  00000  รวมทุกรายการ  2563


In [24]:
thai_months = {
    'ม.ค.': 1, 'ก.พ.': 2, 'มี.ค.': 3, 'เม.ย.': 4, 'พ.ค.': 5, 'มิ.ย.': 6,
    'ก.ค.': 7, 'ส.ค.': 8, 'ก.ย.': 9, 'ต.ค.': 10, 'พ.ย.': 11, 'ธ.ค.': 12
}

records = []
for _, row in cpi_headline.iterrows():
    year_be = int(row['ปี'])
    year_ce = year_be - 543
    for th_month, month_num in thai_months.items():
        value = row[th_month]
        if pd.notna(value) and value != '-':
            records.append({
                'date': pd.Timestamp(year=year_ce, month=month_num, day=1),
                'cpi_index': float(value)
            })

df_cpi = pd.DataFrame(records).sort_values('date').reset_index(drop=True)

df_cpi['cpi_yoy'] = df_cpi['cpi_index'].pct_change(12) * 100
df_cpi['cpi_mom'] = df_cpi['cpi_index'].pct_change(1) * 100

print(df_cpi.shape)
df_cpi.tail(10)

(80, 4)


,date,cpi_index,cpi_yoy,cpi_mom
70,2025-11-01,100.15,-0.486884,0.150000
71,2025-12-01,100.19,-0.278690,0.039940
72,2026-01-01,99.91,-0.656259,-0.279469
73,2026-02-01,99.67,-0.875186,-0.240216
74,2026-03-01,100.27,-0.079721,0.601987
75,2026-04-01,103.03,2.885960,2.752568
76,2026-05-01,103.20,2.788845,0.165000
77,2026-06-01,102.85,2.419837,-0.339147
78,2026-07-01,102.10,1.947079,-0.729217
79,2026-08-01,102.67,2.526463,0.558276


In [25]:
df_cpi.to_csv(f'{PROJECT_PATH}/data/cpi_headline.csv', index=False)

## Section 5: Merge Data

In [26]:
df_full['date'] = pd.to_datetime(df_full['period'])

df_full['mid_rate'] = pd.to_numeric(df_full['mid_rate'], errors='coerce')

df_fx_monthly = df_full.set_index('date')['mid_rate'].resample('MS').mean().reset_index()
df_fx_monthly.columns = ['date', 'fx_mid_rate_avg']

df_policy['announcement_date'] = pd.to_datetime(df_policy['announcement_date'])
date_range = pd.date_range(start=df_cpi['date'].min(), end=df_cpi['date'].max(), freq='MS')
df_policy_monthly = pd.DataFrame({'date': date_range})
df_policy_monthly = pd.merge_asof(df_policy_monthly, df_policy.sort_values('announcement_date'),
                                    left_on='date', right_on='announcement_date', direction='backward')
df_policy_monthly = df_policy_monthly[['date', 'policy_rate']]

df_master = df_cpi.merge(df_fx_monthly, on='date', how='left')
df_master = df_master.merge(df_policy_monthly, on='date', how='left')

print(df_master.shape)
df_master.tail(10)

(80, 6)


,date,cpi_index,cpi_yoy,cpi_mom,fx_mid_rate_avg,policy_rate
70,2025-11-01,100.15,-0.486884,0.150000,32.397805,1.50
71,2025-12-01,100.19,-0.278690,0.039940,31.556865,1.50
72,2026-01-01,99.91,-0.656259,-0.279469,31.271765,1.25
73,2026-02-01,99.67,-0.875186,-0.240216,31.266945,1.25
74,2026-03-01,100.27,-0.079721,0.601987,32.265362,1.00
75,2026-04-01,103.03,2.885960,2.752568,32.349817,1.00
76,2026-05-01,103.20,2.788845,0.165000,32.510495,1.00
77,2026-06-01,102.85,2.419837,-0.339147,32.912455,1.00
78,2026-07-01,102.10,1.947079,-0.729217,33.488633,1.00
79,2026-08-01,102.67,2.526463,0.558276,33.018675,1.00


In [27]:
df_master.to_csv(f'{PROJECT_PATH}/data/master_dataset.csv', index=False)

## Section 5: Oil Price Data (EIA)

In [28]:
import requests

EIA_API_KEY = "XBfkMb67Nb5rV3s5elAyUzfZQSdkiaz8dtj86QeV"

url = "https://api.eia.gov/v2/petroleum/pri/spt/data/"
params = {
    "api_key": EIA_API_KEY,
    "frequency": "monthly",
    "data[0]": "value",
    "facets[series][]": "RBRTE",
    "start": "2020-01",
    "end": "2026-08",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc"
}

response = requests.get(url, params=params)
print(response.status_code)
print(response.text[:1000])

200
{"response":{"total":"80","dateFormat":"YYYY-MM","frequency":"monthly","data":[{"period":"2020-01","duoarea":"ZEU","area-name":"NA","product":"EPCBRENT","product-name":"UK Brent Crude Oil","process":"PF4","process-name":"Spot Price FOB","series":"RBRTE","series-description":"Europe Brent Spot Price FOB (Dollars per Barrel)","value":"63.65","units":"$\/BBL"},{"period":"2020-02","duoarea":"ZEU","area-name":"NA","product":"EPCBRENT","product-name":"UK Brent Crude Oil","process":"PF4","process-name":"Spot Price FOB","series":"RBRTE","series-description":"Europe Brent Spot Price FOB (Dollars per Barrel)","value":"55.66","units":"$\/BBL"},{"period":"2020-03","duoarea":"ZEU","area-name":"NA","product":"EPCBRENT","product-name":"UK Brent Crude Oil","process":"PF4","process-name":"Spot Price FOB","series":"RBRTE","series-description":"Europe Brent Spot Price FOB (Dollars per Barrel)","value":"32.01","units":"$\/BBL"},{"period":"2020-04","duoarea":"ZEU","area-name":"NA","product":"EPCBRENT",

In [29]:
import pandas as pd

data = response.json()
records = data['response']['data']

df_oil = pd.DataFrame(records)
df_oil = df_oil[['period', 'value']].copy()
df_oil.columns = ['date', 'oil_price']

df_oil['date'] = pd.to_datetime(df_oil['date'], format='%Y-%m')
df_oil['oil_price'] = pd.to_numeric(df_oil['oil_price'], errors='coerce')

df_oil = df_oil.sort_values('date').reset_index(drop=True)
print(df_oil.shape)
df_oil.tail(10)

(80, 2)


,date,oil_price
70,2025-11-01,63.80
71,2025-12-01,62.54
72,2026-01-01,66.60
73,2026-02-01,70.89
74,2026-03-01,103.13
75,2026-04-01,117.29
76,2026-05-01,107.14
77,2026-06-01,85.40
78,2026-07-01,83.76
79,2026-08-01,91.08


In [30]:
df_oil.to_csv(f'{PROJECT_PATH}/data/oil_price.csv', index=False)

In [31]:
df_master = df_master.merge(df_oil, on='date', how='left')
print(df_master.shape)
df_master.tail(10)

(80, 7)


,date,cpi_index,cpi_yoy,cpi_mom,fx_mid_rate_avg,policy_rate,oil_price
70,2025-11-01,100.15,-0.486884,0.150000,32.397805,1.50,63.80
71,2025-12-01,100.19,-0.278690,0.039940,31.556865,1.50,62.54
72,2026-01-01,99.91,-0.656259,-0.279469,31.271765,1.25,66.60
73,2026-02-01,99.67,-0.875186,-0.240216,31.266945,1.25,70.89
74,2026-03-01,100.27,-0.079721,0.601987,32.265362,1.00,103.13
75,2026-04-01,103.03,2.885960,2.752568,32.349817,1.00,117.29
76,2026-05-01,103.20,2.788845,0.165000,32.510495,1.00,107.14
77,2026-06-01,102.85,2.419837,-0.339147,32.912455,1.00,85.40
78,2026-07-01,102.10,1.947079,-0.729217,33.488633,1.00,83.76
79,2026-08-01,102.67,2.526463,0.558276,33.018675,1.00,91.08


In [32]:
df_master.to_csv(f'{PROJECT_PATH}/data/master_dataset.csv', index=False)